In [1]:
import pandas as pd #ใช้จัดการข้อมูลตาราง (เปิด CSV,กรองข้อมูล,เซฟไฟล์ Excel)
import requests #ใช้สำหรับ "ดาวน์โหลด" ไฟล์เสียงจากลิงก์ AudioURL ที่อยู่ในตาราง
import librosa #ใช้สำหรับ "วิเคราะห์เสียง" (หาความยาวคลิป และตัดช่วงที่ไม่มีเสียงพูดทิ้ง)
import os #ใช้จัดการระบบไฟล์ (สร้างโฟลเดอร์, รวมที่อยู่ไฟล์, ลบไฟล์ที่ใช้เสร็จแล้ว)
from tqdm import tqdm #ใช้สร้าง "แถบแสดงความคืบหน้า" (Progress Bar) จะได้รู้ว่ารันไปถึงกี่ % แล้ว

In [ ]:
# --- เตรียมความพร้อมของไฟล์ ---

csv_file = 'C:/Users/66982/OneDrive/Desktop/BOTNOI_AutoQC/Email1.csv' 
# ชื่อไฟล์ต้นทางที่มีข้อมูล 7071 แถว

temp_audio_path = 'temp_audio'
# ชื่อโฟลเดอร์ที่จะเอาไว้พักไฟล์เสียงที่เราโหลดมาตรวจ

# เช็กว่ามีโฟลเดอร์สำหรับพักไฟล์เสียงหรือยัง ถ้าไม่มีให้สร้างขึ้นมา
if not os.path.exists(temp_audio_path):
    os.makedirs(temp_audio_path)

# อ่านไฟล์ข้อมูลขึ้นมา โดยระบุ encoding เป็น utf-8-sig เพื่อให้รองรับภาษาไทย
df = pd.read_csv(csv_file, encoding='utf-8-sig')

# แสดงจำนวนแถวทั้งหมดออกมาดู
print(f"จำนวนข้อมูลทั้งหมด: {len(df)} แถว")

# โชว์ตัวอย่างข้อมูล 5 แถวแรกเพื่อตรวจสอบความถูกต้อง
df.head()

In [13]:
# import แล้ว
# --- 1. เตรียมตัว ---
temp_audio_path = 'temp_audio'
if not os.path.exists(temp_audio_path): os.makedirs(temp_audio_path)

test_df_1000 = df.head(1000).copy()
main_results = []

print(f"🚀 เริ่มการตรวจสอบ 1,000 แถวแรก...")

# --- 2. เริ่มลูปรันข้อมูล ---
for index, row in tqdm(test_df_1000.iterrows(), total=len(test_df_1000), desc="QC Processing"):
    try:
        url = row['audio']
        text = str(row['text'])
        ext = url.split('.')[-1].split('?')[0].lower()
        filename = f"audio_{index}.{ext}"
        local_path = os.path.join(temp_audio_path, filename)
        
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            with open(local_path, 'wb') as f:
                f.write(resp.content)
            
            y, sr = librosa.load(local_path, sr=None)
            orig_dur = librosa.get_duration(y=y, sr=sr)
            y_trimmed, _ = librosa.effects.trim(y, top_db=25)
            trim_dur = librosa.get_duration(y=y_trimmed, sr=sr)
            
            word_count = len(text.split())
            ratio = word_count / trim_dur if trim_dur > 0 else 0
            
            # การตัดสิน Status
            if trim_dur < 0.5:
                status = "Abnormal: Too Short"
            elif ratio > 3.5:
                status = "Abnormal: Too Fast"
            elif ratio < 0.5:
                status = "Abnormal: Too Slow"
            else:
                status = "Passed"
                
            res = row.to_dict()
            res.update({
                'audio_index': index,
                'duration_trimmed': round(trim_dur, 2),
                'duration_original': round(orig_dur, 2),
                'word_count': word_count,
                'ratio': round(ratio, 2),
                'qc_status': status,
                'local_path': local_path # เก็บไว้สำหรับกดฟัง
            })
            main_results.append(res)
        else:
            main_results.append({**row.to_dict(), 'audio_index': index, 'qc_status': f'Error: HTTP {resp.status_code}'})
    except Exception as e:
        main_results.append({**row.to_dict(), 'audio_index': index, 'qc_status': f'Error: {str(e)}'})

# --- 3. เซฟผลลัพธ์ลง Excel ---
full_report = pd.DataFrame(main_results)
full_report.to_excel('Main_QC_1000_Rows.xlsx', index=False)

audit_df = full_report[full_report['qc_status'].str.contains('Abnormal', na=False)].copy()
audit_df.to_excel('Audit_Checklist_1000.xlsx', index=False)

print(f"\n✅ ตรวจสอบเสร็จสิ้น! พบสิ่งผิดปกติทั้งหมด {len(audit_df)} แถว")
print("-" * 50)

# --- 4. ส่วนสำหรับกดฟังเสียงไฟล์ที่ผิดปกติ (สุ่มมาให้ดู หรือจะดูทั้งหมดก็ได้) ---
if len(audit_df) > 0:
    print("📢 รายการไฟล์ที่ผิดปกติ:")
    # เลือกมาแสดง 20 อันแรกที่ผิดปกติ (ถ้าอยากดูทั้งหมดให้ลบ .head(20) ออกครับ)
    for idx, row in audit_df.iterrows():
        print(f"📍 ลำดับ (Index): {row['audio_index']} | สถานะ: {row['qc_status']}")
        print(f"📝 ข้อความ: {row['text']}")
        print(f"📊 Ratio: {row['ratio']} | เวลา: {row['duration_trimmed']} วินาที")
        
        # โหลดเสียงมาเล่น (เฉพาะเวอร์ชันที่ Trim แล้ว)
        y_play, sr_play = librosa.load(row['local_path'], sr=None)
        y_play_trim, _ = librosa.effects.trim(y_play, top_db=25)
        ipd.display(ipd.Audio(y_play_trim, rate=sr_play))
        print("-" * 30)
else:
    print("✨ เยี่ยมมาก! ไม่พบไฟล์ที่ผิดปกติใน 1,000 แถวนี้เลยครับ")

🚀 เริ่มการตรวจสอบ 1,000 แถวแรก...


QC Processing: 100%|██████████| 1000/1000 [1:46:55<00:00,  6.42s/it] 



✅ ตรวจสอบเสร็จสิ้น! พบสิ่งผิดปกติทั้งหมด 10 แถว
--------------------------------------------------
📢 รายการไฟล์ที่ผิดปกติ:
📍 ลำดับ (Index): 328 | สถานะ: Abnormal: Too Slow
📝 ข้อความ: nan
📊 Ratio: 0.08 | เวลา: 12.93 วินาที


------------------------------
📍 ลำดับ (Index): 347 | สถานะ: Abnormal: Too Slow
📝 ข้อความ: nan
📊 Ratio: 0.12 | เวลา: 8.49 วินาที


------------------------------
📍 ลำดับ (Index): 463 | สถานะ: Abnormal: Too Slow
📝 ข้อความ: nan
📊 Ratio: 0.19 | เวลา: 5.36 วินาที


...

------------------------------
📍 ลำดับ (Index): 576 | สถานะ: Abnormal: Too Short
📝 ข้อความ: เฮช บี อี เอ เอ็ม แอท ทีโอที ดอท ซีโอ ดอท ทีเอช
📊 Ratio: 23.81 | เวลา: 0.46 วินาที


------------------------------
📍 ลำดับ (Index): 613 | สถานะ: Abnormal: Too Fast
📝 ข้อความ: เจ ไอ อาร์ เอ พี โอ อาร์ เอ็น จุด เฮช อาร์ แอท เอช อาร์ ดอท เน็ต ดอท ซีเอ็น
📊 Ratio: 3.52 | เวลา: 5.12 วินาที


------------------------------
📍 ลำดับ (Index): 616 | สถานะ: Abnormal: Too Fast
📝 ข้อความ: พี โอ เอ็น จี เอส เอ เค ดอท แอล โอ จี ไอ เอส ที ไอ ซี เอส แอท ฮอตเมล ดอท คอม
📊 Ratio: 3.54 | เวลา: 5.93 วินาที


------------------------------
📍 ลำดับ (Index): 624 | สถานะ: Abnormal: Too Fast
📝 ข้อความ: บี ยู เอส เอ บี เอ ดอท เค พลัส ดับบลิว โอ อาร์ เค จุด เอ็ม เอ ไอ แอล แอท เอ็ม อี ดอท เคเคยู ดอท เอซี ดอท ทีเอช
📊 Ratio: 21.94 | เวลา: 1.23 วินาที


------------------------------
